# Demo de punta a punta: un lote del stream a través de las diez etapas

Proyecto EDA 2026-2 · Grupo 1 · Escenario B (comercio electrónico)

Las etapas del pipeline se conectan en lote por archivos: cada notebook lee la salida completa de la etapa anterior en `data/interim/<CATEGORIA>/` y escribe la suya. Este notebook sigue un **lote de 200 registros** de esa cadena, de la etapa 0 a la 9, y dentro de él dos eventos:

| Evento | Qué es | Qué muestra |
| --- | --- | --- |
| `e000001981` | Compra legítima de la orden `o00000313`, que lleva dos productos | Prioridad, estado del producto, índices de precio y de atributos, relacionados, PageRank, comunidad y recomendación final |
| `e000001899` | Primera compra de una ráfaga de una cuenta señalada | La alerta de la etapa 6 y su efecto en las etapas 7, 8 y 9 |

Cómo se recorre cada etapa:

| Etapas | Modo | Qué se usa |
| --- | --- | --- |
| 0, 1 y 2 | En vivo | Los registros crudos que emitió la etapa 0 pasan por la `ColaCircular`, el `Preprocesador` y la `ColaPrioridadEventos` (`src/demo_pipeline.py`) |
| 3 y 6 | En vivo | Los eventos que atiende la etapa 2, en su orden de servicio, actualizan `EtapaAlmacenamiento` y pasan por `DetectorFraude` |
| 4, 5, 7 y 8 | Consulta | Sus índices se construyen con el catálogo o el stream completos, así que la demo los consulta para los productos del lote |
| 9 | En vivo | `ServicioEcommerce` carga las salidas de las etapas 3 a 8 y responde las consultas del escenario B |

En las etapas en vivo, la demo compara su resultado con el que guardó el pipeline completo. Requiere las salidas de los notebooks de las etapas 0 a 9.

## Paso 1. Configuración y lote de entrada

El lote son las posiciones 2 150 a 2 349 del stream que emitió la etapa 0 (`etapa0_lotes.jsonl`). Se eligió porque contiene una ráfaga de fraude, órdenes legítimas de dos productos, duplicados y un registro malformado. La posición y los eventos seguidos están en `src/demo_pipeline.py`.

In [1]:
import sys
from collections import Counter
from dataclasses import asdict
from pathlib import Path

from IPython.display import display
import numpy as np
import pandas as pd

# Raíz del proyecto: se sube de carpeta hasta encontrar la que contiene notebooks/ y data/
RAIZ = Path.cwd()
while not ((RAIZ / "notebooks").is_dir() and (RAIZ / "data").is_dir()) and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))       # para importar src/

from src import config
from src import demo_pipeline as demo

pd.set_option("display.max_colwidth", 70)
COMPRA, FRAUDE = demo.EVENTO_COMPRA, demo.EVENTO_FRAUDE

registros = demo.leer_registros_etapa0()
print(f"Categoría: {config.CATEGORIA}   semilla: {config.SEED}")
print(f"Lote: posiciones {demo.INICIO:,} a {demo.INICIO + len(registros) - 1:,} del stream de la etapa 0")
display(pd.Series(Counter(str(r.get("event_type")) for r in registros), name="registros").to_frame().T)

crudos = {r.get("event_id"): r for r in registros}
print("Registros crudos de los dos eventos seguidos:")
display(pd.DataFrame({"compra": crudos[COMPRA], "fraude": crudos[FRAUDE]}))

Categoría: Office_Products   semilla: 42
Lote: posiciones 2,150 a 2,349 del stream de la etapa 0


,cart,view,purchase,search,review,clik,payment_alert
registros,28,55,45,30,39,1,2


Registros crudos de los dos eventos seguidos:


,compra,fraude
event_id,e000001981,e000001899
event_type,purchase,purchase
user_id,AE3JHPKWMW6MMN6DSGGC2ONYJCJQ,APOFB6P2MCUBESJ7DYTZHQOO37JT
parent_asin,B08FMPDDVJ,B09QHDYTFZ
order_id,o00000313,o00000292
order_type,standard,standard
price,249.99,8.99
payment_id,pay_ffcd646c,pay_b1aa0f6a
device_id,dev_0817a15f,dev_1712fb16
ts,1656702140570,1656702107579


## Paso 2. Etapas 0, 1 y 2: buffer, preprocesamiento y prioridad

- **Etapa 0.** Los registros entran uno por uno a una `ColaCircular` de capacidad 32. Si la cola está llena, el productor espera a que el consumidor retire un micro-lote de hasta 16 registros: hay backpressure, pero no se descarta nada.
- **Etapa 1.** Cada micro-lote pasa por el `Preprocesador`, que valida y normaliza cada registro, descarta los duplicados del lote (merge sort por clave) y de los lotes anteriores (búsqueda binaria en la ventana de claves) y reordena por `ts`.
- **Etapa 2.** Los eventos limpios entran al MinHeap con su clase y su plazo. La etapa atiende hasta 12 eventos por micro-lote y hace una pasada de aging cada dos micro-lotes; al final se vacía el heap.

Productor y consumidor se alternan en un solo hilo, como en la traza del visualizador (`src/trace.py`), para que el resultado sea reproducible.

In [2]:
# 2.1 Recorrido del lote por las etapas 0 a 2
traza = demo.recorrer_etapas_0_a_2(registros)
display(pd.Series(traza["resumen"], name="lote").to_frame().T)
print("Rechazos por motivo:", traza["rechazos_por_motivo"])
print("Atendidos por clase:", traza["atendidos_por_clase"])

for nombre, event_id in (("compra", COMPRA), ("fraude", FRAUDE)):
    print(f"\nPasos de {event_id} ({nombre}):")
    display(pd.DataFrame(traza["pasos"][event_id]).fillna(""))

,registros,micro_lotes,productor_en_espera,rechazados,duplicados,atendidos,ascensos_por_aging
lote,200,13,11,1,12,187,0


Rechazos por motivo: {'tipo_desconocido': 1}
Atendidos por clase: {'3 estándar': 24, '4 programada': 6, '5 navegación': 142, '1 alerta': 1, '2 express': 14}

Pasos de e000001981 (compra):


,etapa,accion,posicion_stream,micro_lote,clase,deadline,orden_servicio,clase_salida,espera_s
0,0,encolado en la ColaCircular,2199.0,,,,,,
1,0,sale de la cola,,4.0,,,,,
2,1,válido y normalizado,,,,,,,
3,2,entra al heap,,,3 estándar,1656788540.57,,,
4,2,atendido,,,,,41.0,3 estándar,23.975



Pasos de e000001899 (fraude):


,etapa,accion,posicion_stream,micro_lote,clase,deadline,orden_servicio,clase_salida,espera_s
0,0,encolado en la ColaCircular,2177.0,,,,,,
1,0,sale de la cola,,2.0,,,,,
2,1,válido y normalizado,,,,,,,
3,2,entra al heap,,,3 estándar,1656788507.579,,,
4,2,atendido,,,,,14.0,3 estándar,3.359


In [3]:
# 2.2 El evento en vivo frente a las salidas guardadas de las etapas 0 y 2
guardados = demo.buscar_en_salidas([COMPRA, FRAUDE])
vivo = {e.event_id: e for e in traza["servidos"]}
compra, fraude = vivo[COMPRA], vivo[FRAUDE]

for nombre, event_id in (("compra", COMPRA), ("fraude", FRAUDE)):
    en_vivo, guardado = asdict(vivo[event_id]), guardados[2][event_id]
    iguales = all(en_vivo[campo] == guardado.get(campo) for campo in en_vivo)
    print(f"\n{event_id} ({nombre}): el evento en vivo es {'idéntico' if iguales else 'distinto'} "
          f"al que guardó la etapa 2")
    display(pd.DataFrame({"crudo (etapa 0)": guardados[0][event_id], "en vivo (etapas 1 y 2)": en_vivo,
                          "guardado (etapa 2)": guardado}))


e000001981 (compra): el evento en vivo es idéntico al que guardó la etapa 2


,crudo (etapa 0),en vivo (etapas 1 y 2),guardado (etapa 2)
event_id,e000001981,e000001981,e000001981
event_type,purchase,purchase,purchase
user_id,AE3JHPKWMW6MMN6DSGGC2ONYJCJQ,AE3JHPKWMW6MMN6DSGGC2ONYJCJQ,AE3JHPKWMW6MMN6DSGGC2ONYJCJQ
parent_asin,B08FMPDDVJ,B08FMPDDVJ,B08FMPDDVJ
order_id,o00000313,o00000313,o00000313
order_type,standard,standard,standard
price,249.99,249.99,249.99
payment_id,pay_ffcd646c,pay_ffcd646c,pay_ffcd646c
device_id,dev_0817a15f,dev_0817a15f,dev_0817a15f
ts,1656702140570,1656702140.57,1656702140.57



e000001899 (fraude): el evento en vivo es idéntico al que guardó la etapa 2


,crudo (etapa 0),en vivo (etapas 1 y 2),guardado (etapa 2)
event_id,e000001899,e000001899,e000001899
event_type,purchase,purchase,purchase
user_id,APOFB6P2MCUBESJ7DYTZHQOO37JT,APOFB6P2MCUBESJ7DYTZHQOO37JT,APOFB6P2MCUBESJ7DYTZHQOO37JT
parent_asin,B09QHDYTFZ,B09QHDYTFZ,B09QHDYTFZ
order_id,o00000292,o00000292,o00000292
order_type,standard,standard,standard
price,8.99,8.99,8.99
payment_id,pay_b1aa0f6a,pay_b1aa0f6a,pay_b1aa0f6a
device_id,dev_1712fb16,dev_1712fb16,dev_1712fb16
ts,1656702107579,1656702107.579,1656702107.579


## Paso 3. Etapa 3: catálogo, caché y estado de los productos

`EtapaAlmacenamiento` recibe los eventos en el orden de servicio de la etapa 2. Para cada producto busca su registro en el catálogo en disco con la `TablaHash` del `AlmacenCatalogo`, con la caché LRU delante, y actualiza su `EstadoProducto` (vistas, carritos, compras y stock) y la matriz usuario-producto que después usa la etapa 7. La demo empieza con el estado vacío, así que cada producto parte de su stock inicial; la última columna es el estado que dejó el stream completo.

In [4]:
# 3.1 Estado de los productos de los dos eventos
from src.etapa_3.storage import PESOS, RUTA_CATALOGO, RUTA_ESTADOS, AlmacenCatalogo, EtapaAlmacenamiento, stock_inicial

almacen = AlmacenCatalogo(RUTA_CATALOGO)
etapa3 = EtapaAlmacenamiento(almacen)
for evento in traza["servidos"]:
    etapa3.procesar(evento)

ASIN = compra.parent_asin
ficha = etapa3.ficha(ASIN)
print(f"Producto de la compra: {ASIN} · {ficha['title'][:80]}")
print(f"Categoría {ficha['categoria_hoja']} · precio de catálogo {ficha['precio']} · tienda {ficha['store']}")

filas = {}
for nombre, evento in (("compra", compra), ("fraude", fraude)):
    estado = etapa3.estado(evento.parent_asin)
    final = pd.read_parquet(RUTA_ESTADOS, filters=[("parent_asin", "==", evento.parent_asin)]).iloc[0]
    filas[f"{nombre}: {evento.parent_asin}"] = {
        "stock inicial": stock_inicial(evento.parent_asin),
        "vistas en el lote": estado.vistas, "carritos en el lote": estado.carritos,
        "compras en el lote": estado.compras, "stock tras el lote": estado.stock,
        "compras en el stream completo": int(final["compras"]), "stock al final del stream": int(final["stock"]),
    }
display(pd.DataFrame(filas))
print(f"Interacción del comprador con el producto en la matriz usuario-producto: "
      f"{etapa3.matriz.valor(compra.user_id, ASIN)} (pesos por tipo de evento: {PESOS})")
print(f"Caché LRU del catálogo: {etapa3.cache.tasa_aciertos():.1%} de aciertos en el lote")

Producto de la compra: B08FMPDDVJ · Hbada Ergonomic Executive Office Chair, Big & Tall Desk Chair, PU Leather Swivel
Categoría Managerial & Executive Chairs · precio de catálogo 249.99 · tienda Hbada


,compra: B08FMPDDVJ,fraude: B09QHDYTFZ
stock inicial,12,35
vistas en el lote,0,0
carritos en el lote,0,0
compras en el lote,1,1
stock tras el lote,11,34
compras en el stream completo,101,44
stock al final del stream,7,27


Interacción del comprador con el producto en la matriz usuario-producto: 5.0 (pesos por tipo de evento: {'view': 1.0, 'cart': 2.0, 'purchase': 4.0, 'review': 1.0, 'return': -4.0})
Caché LRU del catálogo: 29.2% de aciertos en el lote


## Paso 4. Etapa 4: índice de precios (B+Tree en disco)

El B+Tree de la etapa 4 indexa el catálogo por `(precio, parent_asin)` en páginas de 4 KB y guarda, para cada clave, la posición del registro en `catalogo.jsonl`. Una búsqueda exacta lee una página por nivel; una consulta de rango baja hasta la primera hoja y recorre las hojas enlazadas. La etapa 9 usa esa consulta para filtrar por precio (paso 9).

In [5]:
# 4.1 Búsqueda exacta y consulta de rango del producto
from src.etapa_4.bplustree import RUTA_INDICE_PRECIO, ArbolBMas, leer_registro

arbol = ArbolBMas(RUTA_INDICE_PRECIO)
precio = ficha["precio"]
offset, largo = arbol.buscar(precio, ASIN)
lecturas_busqueda = arbol.lecturas
with open(RUTA_CATALOGO, "rb") as archivo:
    registro = leer_registro(archivo, offset, largo)
print(f"Árbol: {len(arbol):,} claves y altura {arbol.altura}")
print(f"buscar({precio}, {ASIN!r}): offset {offset:,} y {largo} bytes en catalogo.jsonl, "
      f"{lecturas_busqueda} páginas leídas")
print(f"Registro en esa posición: {registro['parent_asin']} · {registro['title'][:70]}")

antes = arbol.lecturas
en_rango = list(arbol.rango(0.9 * precio, 1.1 * precio))
print(f"rango({0.9 * precio:.2f}, {1.1 * precio:.2f}): {len(en_rango):,} productos y "
      f"{arbol.lecturas - antes} páginas leídas")
display(pd.DataFrame(en_rango[:5], columns=["precio", "parent_asin", "offset", "largo"]))
arbol.cerrar()

Árbol: 336,415 claves y altura 3
buscar(249.99, 'B08FMPDDVJ'): offset 30,255,338 y 476 bytes en catalogo.jsonl, 3 páginas leídas
Registro en esa posición: B08FMPDDVJ · Hbada Ergonomic Executive Office Chair, Big & Tall Desk Chair, PU Leat
rango(224.99, 274.99): 1,910 productos y 15 páginas leídas


,precio,parent_asin,offset,largo
0,225.0,B00005AKAM,210868283,355
1,225.0,B0002UM0MY,128405877,371
2,225.0,B0007ZGSQI,213917517,337
3,225.0,B000LNE9CG,135074966,434
4,225.0,B000PSWZC8,157921705,401


## Paso 5. Etapa 5: vecinos por atributos (KD-Tree)

La etapa 5 describe cada producto con cuatro atributos normalizados con z-score (logaritmo del precio, calificación media, logaritmo del número de calificaciones y logaritmo de la actividad en el stream) y responde KNN exacto con un KD-Tree por categoría. En la etapa 9 es el respaldo del arranque en frío, para los productos que no tienen señales de comportamiento.

In [6]:
# 5.1 Atributos del producto y sus cinco vecinos más cercanos dentro de su categoría
from src.etapa_5.kdtree import ATRIBUTOS, RUTA_VECTORES, IndiceProductos

z = np.load(RUTA_VECTORES)
indice = IndiceProductos([str(a) for a in z["asins"]], z["Z"] * z["desv"] + z["media"],
                         [str(c) for c in z["nombres_categoria"][z["categoria"]]])
i = indice.posicion.obtener(ASIN)
print("Atributos:", {a: round(float(v), 3) for a, v in zip(ATRIBUTOS, indice.Z[i] * indice.desv + indice.media)})
vecinos_kd = indice.similares(ASIN, k=5)
display(pd.DataFrame([{"distancia": round(float(d), 4), "parent_asin": a, "titulo": (almacen.leer(a) or {}).get("title")}
                      for d, a in vecinos_kd]))

Atributos: {'log_precio': 5.525, 'average_rating': 3.9, 'log_rating_number': 7.074, 'log_actividad_stream': 5.838}


,distancia,parent_asin,titulo
0,0.3911,B09L5G6XFB,Serta Executive Office Adjustable Ergonomic Computer Chair with La...
1,0.5347,B07SJ3HMZJ,Nouhaus ErgoFlip Mesh Computer Chair - Grey Rolling Desk Chair wit...
2,0.5409,B09SDYV1PG,Hbada Office Chair Ergonomic Executive Office Chair PU Leather Swi...
3,0.5500,B07ZCR9NWP,NEO CHAIR Office Chair Computer Desk Chair Gaming - Ergonomic High...
4,0.5675,B09LQ1YRDP,CLATINA Ergonomic Big and Tall Executive Office Chair with Upholst...


## Paso 6. Etapa 6: detección de fraude (Bloom Filter y Count-Min Sketch)

`DetectorFraude` revisa las compras y las alertas de pago en el orden de servicio de la etapa 2:

1. Consulta la cuenta, la tarjeta y el dispositivo del evento en el Bloom Filter de los señalados que conoce (la mitad de cada lista, elegida con la semilla del pipeline) y confirma cada positivo en una `TablaHash` exacta. Una clave confirmada bloquea la orden.
2. Cuenta las órdenes de cada tarjeta en un Count-Min Sketch por ventanas de 5 minutos. Si una tarjeta llega a 5 órdenes, verifica el conteo exacto y retiene la orden para revisión.

La demo crea un detector nuevo para el lote, así que su Count-Min Sketch solo cuenta las órdenes del lote. La tabla compara sus decisiones con las que guardó la etapa 6.

In [7]:
# 6.1 Alertas del lote en vivo frente a las de la etapa 6
from src.etapa_6.probabilistic import FRACCION_CONOCIDA, DetectorFraude, cargar_listas, dividir_listas

conocidas, _ = dividir_listas(cargar_listas(), FRACCION_CONOCIDA, config.SEED)
detector = DetectorFraude(conocidas)
alertas_vivo = {}
for evento in traza["servidos"]:
    alerta = detector.procesar(evento)
    if alerta:
        alertas_vivo[evento.event_id] = alerta
alertas_etapa6 = demo.alertas_guardadas(e.event_id for e in traza["servidos"])

comparacion = pd.DataFrame([
    {"event_id": e, "tipo": vivo[e].event_type, "orden": vivo[e].order_id,
     "decisión en vivo": alertas_vivo.get(e, {}).get("decision", "sin alerta"),
     "motivos en vivo": alertas_vivo.get(e, {}).get("motivos"),
     "decisión de la etapa 6": alertas_etapa6.get(e, {}).get("decision", "sin alerta"),
     "motivos de la etapa 6": alertas_etapa6.get(e, {}).get("motivos")}
    for e in sorted(set(alertas_vivo) | set(alertas_etapa6) | {COMPRA})])
iguales = (comparacion["decisión en vivo"] == comparacion["decisión de la etapa 6"]).sum()
print(f"Eventos revisados en el lote: {detector.metricas['revisados']}; alertas en vivo: {len(alertas_vivo)}; "
      f"alertas de la etapa 6 para estos eventos: {len(alertas_etapa6)}")
print(f"La decisión coincide en {iguales} de {len(comparacion)} eventos")
display(comparacion)

for nombre, evento in (("fraude", fraude), ("compra", compra)):
    clave = f"cuenta:{evento.user_id}"
    print(f"{nombre}: {clave} -> Bloom Filter {clave in detector.bloom}, TablaHash exacta {clave in detector.exacta}")

Eventos revisados en el lote: 45; alertas en vivo: 10; alertas de la etapa 6 para estos eventos: 10
La decisión coincide en 11 de 11 eventos


,event_id,tipo,orden,decisión en vivo,motivos en vivo,decisión de la etapa 6,motivos de la etapa 6
0,e000001899,purchase,o00000292,bloquear,[cuenta],bloquear,[cuenta]
1,e000001901,purchase,o00000294,bloquear,[cuenta],bloquear,"[cuenta, rafaga]"
2,e000001902,purchase,o00000295,bloquear,[cuenta],bloquear,[cuenta]
3,e000001903,purchase,o00000296,bloquear,[cuenta],bloquear,[cuenta]
4,e000001904,payment_alert,o00000296,bloquear,[cuenta],bloquear,[cuenta]
5,e000001905,purchase,o00000297,bloquear,"[cuenta, rafaga]",bloquear,[cuenta]
6,e000001906,purchase,o00000298,bloquear,"[cuenta, rafaga]",bloquear,"[cuenta, rafaga]"
7,e000001907,purchase,o00000299,bloquear,"[cuenta, rafaga]",bloquear,"[cuenta, rafaga]"
8,e000001908,purchase,o00000300,bloquear,"[cuenta, rafaga]",bloquear,"[cuenta, rafaga]"
9,e000001909,purchase,o00000301,bloquear,"[cuenta, rafaga]",bloquear,"[cuenta, rafaga]"


fraude: cuenta:APOFB6P2MCUBESJ7DYTZHQOO37JT -> Bloom Filter True, TablaHash exacta True
compra: cuenta:AE3JHPKWMW6MMN6DSGGC2ONYJCJQ -> Bloom Filter False, TablaHash exacta False


## Paso 7. Etapa 7: relacionados (MinHash + LSH) y similares (HNSW)

La etapa 7 compara los productos de dos formas:

- **Por comportamiento.** El conjunto de usuarios de cada producto sale de la matriz usuario-producto de la etapa 3, sin los usuarios con órdenes bloqueadas por la etapa 6. Cada conjunto se resume en una firma MinHash de 256 hashes y LSH propone los candidatos, cuyo Jaccard se estima con las firmas. Es la señal S_rel de la etapa 9.
- **Por atributos.** Los rasgos del título, la categoría y la tienda forman vectores TF-IDF reducidos a 128 dimensiones, y HNSW encuentra los vecinos. La distancia $d$ se convierte en la señal S_sim de la etapa 9 con $1 - d^2/2$.

In [8]:
# 7.1 Relacionados y similares del producto, y exclusión de la cuenta del fraude
from src.etapa_7.similarity import RUTA_RELACIONADOS, RUTA_SIMILARES, usuarios_bloqueados
from src.etapa_9.service import similitud_desde_distancia


def con_titulo(tabla):
    return tabla.assign(titulo=[(almacen.leer(a) or {}).get("title") for a in tabla["vecino"]])


relacionados = pd.read_parquet(RUTA_RELACIONADOS, filters=[("parent_asin", "==", ASIN)])
similares = pd.read_parquet(RUTA_SIMILARES, filters=[("parent_asin", "==", ASIN)])
similares["s_sim"] = similares["distancia"].map(similitud_desde_distancia)
print(f"Relacionados por comportamiento (MinHash + LSH): {len(relacionados)}")
display(con_titulo(relacionados.head(5)))
print(f"Similares por atributos (HNSW): {len(similares)}")
display(con_titulo(similares.head(5)))

bloqueados = usuarios_bloqueados()
print(f"Usuarios con órdenes bloqueadas por la etapa 6, fuera de los conjuntos de MinHash: {len(bloqueados):,}")
for nombre, evento in (("fraude", fraude), ("compra", compra)):
    print(f"  {nombre}: {evento.user_id} {'excluido' if evento.user_id in bloqueados else 'incluido'}")

Relacionados por comportamiento (MinHash + LSH): 10


,parent_asin,rango,vecino,jaccard,titulo
0,B08FMPDDVJ,1,B09SPX5T3H,0.043478,"Nano Double Sided Tape Heavy Duty, Multipurpose Removable Mounting..."
1,B08FMPDDVJ,2,B07S5YQ16N,0.032520,"Zebra Sarasa Clip 0.5 Retractable Gel Ink Pen, Rubber Grip, 0.5mm,..."
2,B08FMPDDVJ,3,B09XV2BV73,0.024390,"1 inch Round Smile Face Stickers,Happy Face Stickers,Reward Sticke..."
3,B08FMPDDVJ,4,B0B67QCFH3,0.024194,"Calendar 2023-2024 - Wall Calendar 2023-2024, JULY 2023 to DECEMBE..."
4,B08FMPDDVJ,5,B0BMM4XHV8,0.024000,"Paper Mate Gel Pen, Profile Retractable Pen, 0.7mm, Black, 12 Count"


Similares por atributos (HNSW): 10


,parent_asin,rango,vecino,distancia,s_sim,titulo
0,B08FMPDDVJ,1,B09SDYV1PG,0.604712,0.817161,Hbada Office Chair Ergonomic Executive Office Chair PU Leather Swi...
1,B08FMPDDVJ,2,B07CNPB7SH,0.778618,0.696877,"Hbada Office Adjustable Chair, High Back with Breathable Mesh Recl..."
2,B08FMPDDVJ,3,B0BX3PBJSV,0.897262,0.597461,"Ergonomic Office Desk Chair, Mid-Back Mesh Computer Chair, Height ..."
3,B08FMPDDVJ,4,B08XZ8SD5Z,0.936313,0.561659,"Mimoglad Office Chair, Ergonomic Desk Chair with Adjustable Lumbar..."
4,B08FMPDDVJ,5,B0BWY9F2CR,0.958915,0.540241,"Guessky Office Chair, Executive Office Chair Big and Tall Office C..."


Usuarios con órdenes bloqueadas por la etapa 6, fuera de los conjuntos de MinHash: 30
  fraude: APOFB6P2MCUBESJ7DYTZHQOO37JT excluido
  compra: AE3JHPKWMW6MMN6DSGGC2ONYJCJQ incluido


## Paso 8. Etapa 8: grafo de co-compra, PageRank y comunidades

La etapa 8 arma el grafo con las órdenes de dos o más productos del stream de la etapa 2, sin las órdenes bloqueadas por la etapa 6: el peso de una arista es el número de órdenes en que se compraron juntos sus dos productos. La orden de la compra seguida aporta una de esas aristas. Sobre el grafo se calculan PageRank, la señal S_pr de la etapa 9, y las comunidades de Louvain; el peso de la arista, dividido por el mayor peso del ancla, es la señal S_co.

In [9]:
# 8.1 La arista de la orden, el PageRank y la comunidad del producto
from src.etapa_8.graph import RUTA_COMUNIDADES, RUTA_GRAFO, RUTA_PAGERANK, GrafoCoCompra, ordenes_bloqueadas

grafo = GrafoCoCompra.cargar_npz(RUTA_GRAFO)
vecinos, pesos = grafo.vecinos(grafo.posicion.obtener(ASIN))
peso_con = {grafo.asins[j]: float(w) for j, w in zip(vecinos.tolist(), pesos.tolist())}
otros = sorted({e.parent_asin for e in traza["servidos"]
                if e.order_id == compra.order_id and e.event_type == "purchase"} - {ASIN})
print(f"Orden {compra.order_id}: {ASIN} junto con {', '.join(otros)}")
for otro in otros:
    print(f"  arista {ASIN} - {otro}: {peso_con.get(otro, 0):.0f} órdenes juntas en el stream completo")
print(f"{ASIN} tiene {len(peso_con)} vecinos de co-compra y grado ponderado {sum(peso_con.values()):.0f}")

pr = pd.read_parquet(RUTA_PAGERANK, filters=[("parent_asin", "==", ASIN)]).iloc[0]
com = pd.read_parquet(RUTA_COMUNIDADES, filters=[("parent_asin", "==", ASIN)]).iloc[0]
print(f"PageRank {pr['pagerank_score']:.6f}: puesto {int(pr['rango_pagerank']):,} de {grafo.n:,} productos")
print(f"Comunidad de Louvain {int(com['comunidad_louvain'])}, con {int(com['tamano_comunidad_louvain'])} productos")

bloqueadas = ordenes_bloqueadas()
print(f"Orden del fraude {fraude.order_id}: {'excluida' if fraude.order_id in bloqueadas else 'incluida'} "
      f"({len(bloqueadas):,} órdenes bloqueadas por la etapa 6 quedan fuera del grafo)")

Orden o00000313: B08FMPDDVJ junto con B09SPX5T3H
  arista B08FMPDDVJ - B09SPX5T3H: 10 órdenes juntas en el stream completo
B08FMPDDVJ tiene 63 vecinos de co-compra y grado ponderado 118
PageRank 0.000045: puesto 2,352 de 150,534 productos
Comunidad de Louvain 153, con 279 productos
Orden del fraude o00000292: excluida (31,994 órdenes bloqueadas por la etapa 6 quedan fuera del grafo)


## Paso 9. Etapa 9: servicio y salida observable

`ServicioEcommerce` carga el catálogo y el estado de la etapa 3, el B+Tree de la etapa 4, el KD-Tree de la etapa 5, las alertas de la etapa 6, los vecinos de la etapa 7 y el grafo de la etapa 8, e indexa los títulos en el Trie radix y en el índice invertido. Para la compra seguida responde lo que vería el cliente: la ficha del producto, el autocompletado de su marca y la recomendación híbrida, que elige el Top-K con un MinHeap, sin filtro y con un filtro por precio. Para el fraude, el centro de monitoreo del dashboard muestra la alerta.

In [10]:
# 9.1 Carga del servicio, ficha del producto y autocompletado de su marca
from src.etapa_9.service import ServicioEcommerce, normalizar_texto

servicio = ServicioEcommerce(config.CARPETA_INTERIM.parent, categoria=config.CATEGORIA)
servicio.cargar_datos()
detalle = servicio.detalle_producto(ASIN)
display(pd.Series({c: detalle.get(c) for c in ("title", "store", "categoria_hoja", "precio", "stock", "disponible",
                                               "pagerank", "comunidad_louvain")}, name=ASIN).to_frame())
prefijo = normalizar_texto(detalle.get("store") or detalle["title"].split()[0])
print(f"Autocompletado de {prefijo!r}:")
display(pd.DataFrame(servicio.autocompletar(prefijo, k=5)))

ServicioEcommerce inicializado en 125.24 s


,B08FMPDDVJ
title,"Hbada Ergonomic Executive Office Chair, Big & Tall Desk Chair, PU ..."
store,Hbada
categoria_hoja,Managerial & Executive Chairs
precio,249.99
stock,7
disponible,True
pagerank,0.000045
comunidad_louvain,153


Autocompletado de 'hbada':


,termino,frecuencia,meta
0,Hbada,500.000000,"{'tipo': 'marca', 'asin': 'B08FMPDDVJ'}"
1,Hbada Ergonomic Executive Office,12.252864,"{'tipo': 'producto', 'asin': 'B08FMPDDVJ', 'titulo': 'Hbada Ergono..."
2,Hbada Office Chair Ergonomic,10.992883,"{'tipo': 'producto', 'asin': 'B09SDYV1PG', 'titulo': 'Hbada Office..."
3,"Hbada Office Adjustable Chair,",3.522965,"{'tipo': 'producto', 'asin': 'B07CNPB7SH', 'titulo': 'Hbada Office..."
4,Hbada Office Task Desk,3.522965,"{'tipo': 'producto', 'asin': 'B07WSSX5Q5', 'titulo': 'Hbada Office..."


In [11]:
# 9.2 Recomendación híbrida del producto, sin filtro y con precio entre la mitad y el precio del producto
columnas = ["rango", "parent_asin", "score_hibrido", "similitud_atributos", "jaccard_comportamiento",
            "afinidad_cocompra", "pagerank", "precio", "stock", "titulo"]
print("Señales: similitud_atributos = S_sim (HNSW, etapa 7) · jaccard_comportamiento = S_rel (MinHash + LSH, etapa 7) · "
      "afinidad_cocompra = S_co y pagerank = S_pr (etapa 8) · stock (etapa 3)")
recs = pd.DataFrame(servicio.recomendar(ASIN, k=8), columns=columnas)
display(recs)
recs_precio = pd.DataFrame(servicio.recomendar(ASIN, k=5, precio_min=0.5 * precio, precio_max=precio), columns=columnas)
print(f"Con precio entre ${0.5 * precio:.2f} y ${precio:.2f} (B+Tree de la etapa 4):")
display(recs_precio)

candidatos = servicio.puntuar_candidatos(ASIN)
orden_candidatos = sorted(candidatos, key=lambda a: -candidatos[a][0])
puesto_otro = {}
for otro in otros:
    if otro in candidatos:
        puesto_otro[otro] = orden_candidatos.index(otro) + 1
        print(f"{otro}, de la misma orden, es candidato por co-compra: S_co = {candidatos[otro][1]['co']:.3f}, "
              f"puesto {puesto_otro[otro]} de {len(candidatos)} candidatos")

Señales: similitud_atributos = S_sim (HNSW, etapa 7) · jaccard_comportamiento = S_rel (MinHash + LSH, etapa 7) · afinidad_cocompra = S_co y pagerank = S_pr (etapa 8) · stock (etapa 3)


,rango,parent_asin,score_hibrido,similitud_atributos,jaccard_comportamiento,afinidad_cocompra,pagerank,precio,stock,titulo
0,1,B09SDYV1PG,0.8189,0.8172,0.0000,0.0,0.000040,169.99,12,Hbada Office Chair Ergonomic Executive Office Chair PU Leather Swi...
1,2,B07CNPB7SH,0.7025,0.6969,0.0000,0.0,0.000000,NaN,36,"Hbada Office Adjustable Chair, High Back with Breathable Mesh Recl..."
2,3,B09SPX5T3H,0.6849,0.0000,0.0435,1.0,0.000053,6.99,3,"Nano Double Sided Tape Heavy Duty, Multipurpose Removable Mounting..."
3,4,B0BX3PBJSV,0.6586,0.5975,0.0000,0.0,0.000007,NaN,43,"Ergonomic Office Desk Chair, Mid-Back Mesh Computer Chair, Height ..."
4,5,B0BWY9F2CR,0.6233,0.5402,0.0000,0.0,0.000008,179.99,25,"Guessky Office Chair, Executive Office Chair Big and Tall Office C..."
5,6,B08XZ8SD5Z,0.6184,0.5617,0.0000,0.0,0.000001,NaN,12,"Mimoglad Office Chair, Ergonomic Desk Chair with Adjustable Lumbar..."
6,7,B0B76DTV37,0.6168,0.5270,0.0000,0.0,0.000009,NaN,33,"PU Leather Home Office Chair, Tall and Big High Back Ergonomic Exe..."
7,8,B0B631Q3FM,0.6131,0.5349,0.0000,0.0,0.000004,NaN,8,"Ergonomic Home Office Chair, High Back PU Leather Executive Comput..."


Con precio entre $125.00 y $249.99 (B+Tree de la etapa 4):


,rango,parent_asin,score_hibrido,similitud_atributos,jaccard_comportamiento,afinidad_cocompra,pagerank,precio,stock,titulo
0,1,B09SDYV1PG,0.8189,0.8172,0.0,0.0,0.000040,169.99,12,Hbada Office Chair Ergonomic Executive Office Chair PU Leather Swi...
1,2,B0BWY9F2CR,0.6233,0.5402,0.0,0.0,0.000008,179.99,25,"Guessky Office Chair, Executive Office Chair Big and Tall Office C..."
2,3,B09VXLN75P,0.3037,0.0000,0.0,0.1,0.000003,139.59,15,LVELIMIT C3224 C3210K0 Toner Cartridge Replacement for MC3224i MC3...
3,4,B01N32L1GX,0.3034,0.0000,0.0,0.1,0.000003,127.99,40,HOMCOM Multi-Function Computer Desk Home Office Workstation with K...


B09SPX5T3H, de la misma orden, es candidato por co-compra: S_co = 1.000, puesto 3 de 73 candidatos


In [12]:
# 9.3 La alerta del fraude en el centro de monitoreo del dashboard
resumen_f = servicio.resumen_fraude()
print({k: resumen_f["resumen"][k] for k in ("total_alertas", "bloqueadas", "en_revision")})
display(pd.DataFrame(resumen_f["muestra"]).query("event_id == @FRAUDE"))
servicio.cerrar()
almacen.cerrar()

{'total_alertas': 47710, 'bloqueadas': 41532, 'en_revision': 6178}


,event_id,order_id,user_id,decision,motivos,fraude_real,estimado_cms
0,e000001899,o00000292,APOFB6P2MCUBESJ7DYTZHQOO37JT,bloquear,[cuenta],True,1.0


## Resumen de la traza

In [13]:
# Qué hizo cada etapa con los dos eventos
def paso(event_id, accion):
    return next(p for p in traza["pasos"][event_id] if p["accion"] == accion)


def etapa1(event_id):
    return ", ".join(p["accion"] for p in traza["pasos"][event_id] if p["etapa"] == 1)


def decision(event_id):
    alerta = alertas_vivo.get(event_id)
    return f"{alerta['decision']} ({', '.join(alerta['motivos'])})" if alerta else "sin alerta"


resumen_traza = pd.DataFrame({
    f"compra {COMPRA}": {
        "0 · ColaCircular": f"posición {paso(COMPRA, 'encolado en la ColaCircular')['posicion_stream']:,}, "
                            f"micro-lote {paso(COMPRA, 'sale de la cola')['micro_lote']}",
        "1 · Preprocesador": etapa1(COMPRA),
        "2 · MinHeap": f"clase {paso(COMPRA, 'entra al heap')['clase']}, atendido en el puesto "
                       f"{paso(COMPRA, 'atendido')['orden_servicio']}",
        "3 · TablaHash, LRU y estado": f"stock {stock_inicial(ASIN)} -> {etapa3.estado(ASIN).stock} en el lote",
        "4 · B+Tree": f"clave ({precio}, {ASIN}) en {lecturas_busqueda} páginas; {len(en_rango):,} productos a ±10 %",
        "5 · KD-Tree": f"{len(vecinos_kd)} vecinos por atributos en {ficha['categoria_hoja']}",
        "6 · Bloom Filter y CMS": decision(COMPRA),
        "7 · MinHash + LSH y HNSW": f"{len(relacionados)} relacionados y {len(similares)} similares",
        "8 · Grafo, PageRank y Louvain": f"arista con {', '.join(otros)}; PageRank puesto {int(pr['rango_pagerank']):,}; "
                                         f"comunidad {int(com['comunidad_louvain'])}",
        "9 · Servicio": f"{len(recs)} recomendaciones; "
                        + ", ".join(f"{a} es el candidato {k} de {len(candidatos)}" for a, k in puesto_otro.items()),
    },
    f"fraude {FRAUDE}": {
        "0 · ColaCircular": f"posición {paso(FRAUDE, 'encolado en la ColaCircular')['posicion_stream']:,}, "
                            f"micro-lote {paso(FRAUDE, 'sale de la cola')['micro_lote']}",
        "1 · Preprocesador": etapa1(FRAUDE),
        "2 · MinHeap": f"clase {paso(FRAUDE, 'entra al heap')['clase']}, atendido en el puesto "
                       f"{paso(FRAUDE, 'atendido')['orden_servicio']}",
        "3 · TablaHash, LRU y estado": f"stock {stock_inicial(fraude.parent_asin)} -> "
                                       f"{etapa3.estado(fraude.parent_asin).stock}: la etapa 3 cuenta la compra antes del filtro de fraude",
        "4 · B+Tree": "—",
        "5 · KD-Tree": "—",
        "6 · Bloom Filter y CMS": decision(FRAUDE),
        "7 · MinHash + LSH y HNSW": "cuenta excluida de los conjuntos" if fraude.user_id in bloqueados else "cuenta incluida",
        "8 · Grafo, PageRank y Louvain": "orden excluida del grafo" if fraude.order_id in bloqueadas else "orden en el grafo",
        "9 · Servicio": "alerta en el centro de monitoreo del dashboard",
    },
})
display(resumen_traza)

,compra e000001981,fraude e000001899
0 · ColaCircular,"posición 2,199, micro-lote 4","posición 2,177, micro-lote 2"
1 · Preprocesador,válido y normalizado,válido y normalizado
2 · MinHeap,"clase 3 estándar, atendido en el puesto 41","clase 3 estándar, atendido en el puesto 14"
"3 · TablaHash, LRU y estado",stock 12 -> 11 en el lote,stock 35 -> 34: la etapa 3 cuenta la compra antes del filtro de fr...
4 · B+Tree,"clave (249.99, B08FMPDDVJ) en 3 páginas; 1,910 productos a ±10 %",—
5 · KD-Tree,5 vecinos por atributos en Managerial & Executive Chairs,—
6 · Bloom Filter y CMS,sin alerta,bloquear (cuenta)
7 · MinHash + LSH y HNSW,10 relacionados y 10 similares,cuenta excluida de los conjuntos
"8 · Grafo, PageRank y Louvain","arista con B09SPX5T3H; PageRank puesto 2,352; comunidad 153",orden excluida del grafo
9 · Servicio,8 recomendaciones; B09SPX5T3H es el candidato 3 de 73,alerta en el centro de monitoreo del dashboard


### Lectura de la traza

- **Etapas 0 a 2.** De los 200 registros, la etapa 1 rechaza 1, el tipo de evento desconocido `clik`, y descarta 12 duplicados; la etapa 2 atiende los 187 eventos restantes. La cola se llenó 11 veces y el productor esperó sin perder registros. Las dos compras entran en la clase 3 (estándar), con un plazo de 24 h, y el evento que sale de la etapa 2 en vivo es idéntico, campo por campo, al que guardó el pipeline completo: `ts` en segundos, categoría normalizada y precio como número.
- **Etapa 3.** La compra baja el stock de la silla de 12 a 11 y suma 5 a la celda del comprador en la matriz usuario-producto: 4 por la compra y 1 por su reseña `e000001984`, que viene en el mismo lote. Al final del stream la silla lleva 101 compras y le quedan 7 unidades, el stock que muestra la ficha del servicio en el paso 9. La compra fraudulenta también descuenta stock, porque la etapa 3 se actualiza antes del filtro de la etapa 6.
- **Etapas 4 y 5.** El B+Tree encuentra la clave `(249.99, B08FMPDDVJ)` leyendo 3 páginas, una por nivel, y la consulta de ±10 % devuelve 1 910 productos en 15 páginas. El KD-Tree devuelve cinco sillas de la misma categoría, la más cercana a distancia 0.39.
- **Etapa 6.** Las 10 alertas en vivo son bloqueos de la ráfaga de la cuenta señalada, que da positivo en el Bloom Filter y se confirma en la `TablaHash`. La decisión coincide con la de la etapa 6 en los 11 eventos comparados. Los motivos de ráfaga cambian en dos eventos porque el pipeline completo atendió `e000001901` después de `e000001905`, así que la quinta orden de la tarjeta, la que dispara la ráfaga, es otra.
- **Etapa 7.** El relacionado más fuerte de la silla por comportamiento es la cinta `B09SPX5T3H` de la misma orden (Jaccard 0.043), y sus similares por atributos son otras sillas, dos de ellas de Hbada (S_sim 0.82 y 0.70). La cuenta del fraude es uno de los 30 usuarios que la etapa 7 deja fuera de los conjuntos de MinHash.
- **Etapa 8.** La orden `o00000313` aporta la arista entre la silla y la cinta, que suma 10 órdenes en el stream completo. La silla tiene 63 vecinos de co-compra y su PageRank está en el puesto 2 352 de 150 534; su comunidad de Louvain, la 153, tiene 279 productos. La orden del fraude es una de las 31 994 órdenes bloqueadas que no entran al grafo.
- **Etapa 9.** La recomendación combina las fuentes: primero quedan las dos sillas Hbada más parecidas por atributos, y tercera la cinta de la misma orden, sin similitud de atributos pero con S_co = 1 y S_rel = 0.043. Con el filtro de $125 a $249.99 quedan 4 de los 5 pedidos, porque solo 4 de los 73 candidatos tienen precio en ese rango; las sillas sin precio en el catálogo no están en el B+Tree y nunca pasan el filtro. El KD-Tree de la etapa 5 solo interviene en el arranque en frío, que esta compra no necesita. La alerta del fraude es la primera de la muestra del centro de monitoreo, entre los 41 532 bloqueos de la etapa 6. La carga del servicio tarda 125 s.

En resumen, el lote atraviesa las diez etapas y deja una salida observable en cada una. Donde la demo corre en vivo reproduce lo que guardó el pipeline completo, y la recomendación final usa señales de las etapas 3, 4, 7 y 8.